In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import proplot as pplt
sys.path.insert(0,'..')
from scripts.utils import Config
from scripts.models.sr.train import load_data,subsample_timestep
warnings.filterwarnings('ignore')
pplt.rc.update({'figure.dpi':100})

In [ ]:
config    = Config()
RUNNAME   = 'sr_gauss_all'
RUNCONFIG = config.sr['runs'][RUNNAME]
SEEDS     = config.sr['seeds']
SUBFRAC   = config.sr['subsetfrac']
SEED      = SEEDS[0]

with open(os.path.join(config.splitsdir,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
TPMEAN = STATS[f'{config.targetvar}_mean']
TPSTD  = STATS[f'{config.targetvar}_std']

In [ ]:
def load_split_data():
    xtrain,ytrain,reftrain,vmtrain = load_data('train',RUNCONFIG,config,time_offset=0)
    xvalid,yvalid,_,vmvalid       = load_data('valid',RUNCONFIG,config,time_offset=int(reftrain.sizes['time']))
    xfit = pd.concat([xtrain[vmtrain],xvalid[vmvalid]]).reset_index(drop=True)
    yfit = np.concatenate([ytrain[vmtrain],yvalid[vmvalid]])
    return xfit,yfit

def to_mm(y):
    return np.expm1(y*TPSTD+TPMEAN)

def bin_fracs(tp,bins):
    counts = np.array([(tp<=bins[1]).sum()]+
                      [((tp>bins[i])&(tp<=bins[i+1])).sum() for i in range(1,len(bins)-1)])
    return counts/counts.sum()

In [ ]:
xfit,yfit = load_split_data()
xsub,ysub = subsample_timestep(xfit,yfit,subsetfrac=SUBFRAC,seed=SEED)
tp_full   = to_mm(yfit)
tp_sub    = to_mm(ysub)
print(f'{RUNNAME} | full: {len(tp_full):,} | subset: {len(tp_sub):,} | frac: {len(tp_sub)/len(tp_full):.3%} (requested: {SUBFRAC:.3%})')

In [ ]:
def summary_stats(tp):
    return {'n':len(tp),'dry (≤1e-4 mm)':f'{np.mean(tp<=1e-4):.2%}',
            'mean':f'{np.mean(tp):.4f}','median':f'{np.median(tp):.4f}',
            'p90':f'{np.percentile(tp,90):.4f}','p99':f'{np.percentile(tp,99):.4f}',
            'max':f'{np.max(tp):.4f}'}

pd.DataFrame({'full train+valid':summary_stats(tp_full),f'sr subset (seed {SEED})':summary_stats(tp_sub)}).T

In [ ]:
BINS   = np.array([0,1e-4,1e-2,1e-1,1,10,100,np.inf])
LABELS = ['dry (≤1e-4)','(1e-4, 1e-2]','(1e-2, 0.1]','(0.1, 1]','(1, 10]','(10, 100]','>100']

pd.DataFrame({'label':LABELS,
              'full train+valid':bin_fracs(tp_full,BINS),
              f'sr subset (seed {SEED})':bin_fracs(tp_sub,BINS)})

In [ ]:
wet_full = tp_full[tp_full>1e-4]
wet_sub  = tp_sub[tp_sub>1e-4]
bins     = np.logspace(-4,2,80)
fig,ax   = pplt.subplots(refwidth=5,refheight=2.5)
ax.format(xlabel='Total Precipitation (mm)',xscale='log',xformatter='log',
          ylabel='Density',title=f'{RUNNAME}: wet-point precipitation distribution',grid=True)
ax.hist(wet_full,bins=bins,density=True,histtype='step',linewidth=1.5,color='#1B2C61',label='Full train+valid')
ax.hist(wet_sub, bins=bins,density=True,histtype='step',linewidth=1.5,color='#D42028',label=f'SR subset (seed {SEED})')
ax.legend(loc='ul',ncols=1)
pplt.show()

In [ ]:
xidx = np.arange(len(LABELS))
fig,ax = pplt.subplots(refwidth=6,refheight=2.5)
ax.format(xlabel='Precipitation Bin',ylabel='Fraction of Samples',
          title=f'{RUNNAME}: rain-bin fractions',grid=True,
          xlocator=xidx,xformatter=LABELS)
ax.plot(xidx,bin_fracs(tp_full,BINS),color='#1B2C61',marker='o',label='Full train+valid')
ax.plot(xidx,bin_fracs(tp_sub,BINS), color='#D42028',marker='o',label=f'SR subset (seed {SEED})')
ax.legend(loc='ur',ncols=1)
pplt.show()